# AI TRAINING – WEEK 4 • DAY 2
## Machine Learning Workflow & Data Preprocessing
### Building Robust, Leak-Free Scikit-Learn Pipelines with ColumnTransformer

---

- **Student Name:** Lasya Kolluru
- **Email:** kollurulasyachowdary@gmail.com
- **Date:** 27-Oct-2026
- **Duration:** 5 hours
- **Difficulty:** Intermediate
- **Track:** Machine Learning Foundations
- **Deliverable:** `Week4_Day2_Preprocessing_Lasya_Kolluru.ipynb` & `customer_churn_small.csv`
- **Assessment Total:** 100 Marks

---

### Project Overview
In real-world machine learning, building predictive models requires far more than applying algorithms like Logistic Regression or Random Forests. The most crucial phase is **Data Preprocessing**: identifying missing entries, standardizing numerical features, encoding non-numeric categorical attributes, and designing an architecture that eliminates **Data Leakage**.

This notebook provides an exhaustive, hands-on implementation of the complete ML workflow using **Scikit-Learn** (`Pipeline`, `ColumnTransformer`, `SimpleImputer`, `StandardScaler`, and `OneHotEncoder`). Working with the sample customer churn dataset (`customer_churn_small.csv`), we demonstrate why preprocessing parameters must be learned **strictly from training data** and merely applied to test data. We also debug common real-world errors and answer core conceptual questions.

---
## 1. Learning Goals

By the end of this assignment, the following competencies are achieved:
1. **ML Workflow Architecture:** Describe each stage from business problem definition, data ingestion, EDA, train/test splitting, preprocessor fitting, model training, held-out evaluation, to iterative improvement.
2. **Data Diagnostics:** Detect and analyze missing values (`NaN`), duplicate rows, inconsistent categories, and uninformative or target-leaking candidate columns.
3. **Feature Scaling Mechanics:** Explain the mathematical rationale for numerical scaling (Standardization vs. Normalization), identify which algorithms strictly require scaling, and explain why tree-based models do not.
4. **Categorical Encoding:** Encode nominal categories into model-friendly binary vectors via One-Hot Encoding while preventing spurious numerical ordinality and handling unseen categories gracefully.
5. **Data Leakage Prevention:** Prevent test data contamination by isolating the train/test split *before* fitting any preprocessing steps (imputers, scalers, encoders).
6. **Reusable Scikit-Learn Pipelines:** Construct a production-grade, modular `Pipeline` using `ColumnTransformer` to orchestrate parallel preprocessing streams and an end-to-end predictive estimator.

---
## 2. Concepts in Simple Words

| Concept | Simple Definition | Practical Purpose |
| :--- | :--- | :--- |
| **ML Workflow** | A structured 8-stage sequence: Business Framing $\to$ Data Collection $\to$ Inspection & Cleaning $\to$ Train/Test Splitting $\to$ Preprocessor Fitting $\to$ Model Training $\to$ Held-out Evaluation $\to$ Model Deployment. | Ensures methodical, reproducible, and leak-free machine learning development. |
| **Data Preprocessing** | Transforming raw, messy tabular data into clean, complete numerical matrices ready for mathematical algorithms. | Real models cannot process missing entries or arbitrary raw text strings directly. |
| **Missing Values (`NaN`)** | Empty or unrecorded cells in a dataset. Handled via median imputation for numerical data and mode / most-frequent imputation for categorical data. | Prevents model errors while preserving sample rows without introducing statistical bias. |
| **Feature Scaling** | Standardizing numerical features onto comparable numerical scales (e.g., mean 0, variance 1 with `StandardScaler`). | Prevents large-scale features (like spend: 500-2500) from dominating small-scale features (like tickets: 0-5). |
| **Categorical Encoding** | Converting non-numeric labels (e.g., city names, plan tiers) into binary numeric indicators using One-Hot Encoding. | Eliminates false mathematical ordering (e.g., assigning Chennai=1, Hyderabad=2 implying Hyderabad is twice Chennai). |
| **Data Leakage** | When information from the test dataset or the future target variable contaminates model training. | Causes deceptively high training/validation scores that fail drastically in real-world deployment. |
| **Pipeline** | A sequential scikit-learn container chaining preprocessing transformers and an estimator into a single atomic object. | Guarantees identical transformations across train and test sets with zero manual repetitive code. |
| **`fit` vs `transform`** | `fit()` learns rules/statistics (medians, means, standard deviations, category lists) from data; `transform()` applies those learned rules. | Training uses `fit_transform()`; testing uses `transform()` only to prevent leakage. |

---
## 3. Suggested Five-Hour Plan

| Time Window | Planned Activity | Key Deliverables & Focus |
| :--- | :--- | :--- |
| **Hour 1** | Understand Workflow & Inspect Sample Dataset | Review the ML workflow; create and load `customer_churn_small.csv`; run initial structural inspections (`head`, `shape`, `info`). |
| **Hour 2** | Diagnostics & Target Definition | Check missing values, identify duplicates, analyze target distribution (`churn`), and separate candidate features ($X$) from target ($y$). |
| **Hour 3** | Numerical & Categorical Preprocessing Strategy | Study median imputation, standard scaling, and one-hot encoding; verify why city codes must not be arbitrarily mapped to integers. |
| **Hour 4** | Build ColumnTransformer & Full Pipeline | Split data using `train_test_split(stratify=y)`; build sub-pipelines and combine them with `ColumnTransformer`; fit preprocessor on train only. |
| **Hour 5** | Debugging Labs, Conceptual Explanations & Review | Complete 5 debugging tasks; answer own-words questions; analyze evaluation metrics and dataset size limitations. |

---
## 4. Sample Dataset Specification & Ingestion

We work with the provided 12-row customer churn dataset (`customer_churn_small.csv`). 

### Dataset Schema
- **`age`**: Customer age in years (Numeric, contains missing values).
- **`monthly_spend`**: Monthly bill amount in INR (Numeric, contains missing values).
- **`city`**: Customer location (Categorical: Hyderabad, Chennai, Bengaluru, Visakhapatnam).
- **`plan_type`**: Subscription tier (Categorical: Basic, Standard, Premium).
- **`support_tickets`**: Number of support requests logged (Numeric, contains missing values).
- **`churn`**: Target label indicating whether customer cancelled service (`Yes` or `No`).

> **Trainer Note:** This tiny dataset is designed specifically for **workflow and leakage prevention mastery**. Because $N=12$, the quantitative evaluation metrics (e.g., accuracy) are demonstrative rather than statistically generalizable.

In [1]:
# Setup runtime environment and import core libraries
import os
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Python version:       ", sys.version.split()[0])
print("Pandas version:       ", pd.__version__)
print("NumPy version:        ", np.__version__)
print("Scikit-Learn version: ", sklearn.__version__)


Python version:        3.14.5
Pandas version:        3.0.6
NumPy version:         2.5.3
Scikit-Learn version:  1.9.1


In [2]:
# Create and persist customer_churn_small.csv if not already present
dataset_path = "customer_churn_small.csv"

raw_csv_data = """age,monthly_spend,city,plan_type,support_tickets,churn
24,500,Hyderabad,Basic,2,Yes
35,1200,Chennai,Premium,0,No
29,,Hyderabad,Standard,1,No
42,2200,Bengaluru,Premium,3,Yes
31,900,Chennai,Basic,,No
,,Hyderabad,Basic,4,Yes
27,750,Visakhapatnam,Standard,1,No
39,1800,Bengaluru,Premium,2,No
33,1100,Chennai,Standard,0,No
46,2500,Hyderabad,Premium,5,Yes
28,650,Bengaluru,Basic,1,No
36,1500,Chennai,Standard,2,Yes"""

with open(dataset_path, "w", encoding="utf-8") as f:
    f.write(raw_csv_data.strip())

print(f"Dataset '{dataset_path}' successfully saved and verified. File size: {os.path.getsize(dataset_path)} bytes.")


Dataset 'customer_churn_small.csv' successfully saved and verified. File size: 420 bytes.


---
## 5. Step-by-Step Practical Implementation

We execute all 10 steps in exact sequential order as specified in the assignment guidelines.

### Step 1 — Load and Inspect Data
We load `customer_churn_small.csv` into a pandas DataFrame and perform comprehensive exploratory checks:
- Display the complete tabular data (`head(12)`)
- Inspect dataset dimensions (`shape`)
- Review schema, memory, and non-null counts (`info()`)
- Tally missing values per column (`isna().sum()`)
- Check for exact row duplicates (`duplicated().sum()`)
- Examine the class balance of the target variable (`churn`)

In [3]:
# Step 1: Load and Inspect Dataset
df = pd.read_csv("customer_churn_small.csv")

print("=== 1. Complete Dataset Inspection (12 Rows) ===")
print(df.to_string())

print("\n=== 2. Dataset Dimensions ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n=== 3. DataFrame Information ===")
df.info()

print("\n=== 4. Missing Values Analysis ===")
missing_series = df.isnull().sum()
missing_pct = (missing_series / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing_series, "Percentage (%)": missing_pct.round(2)})
print(missing_df)

print("\n=== 5. Duplicate Records Check ===")
dup_count = df.duplicated().sum()
print(f"Total duplicate rows detected: {dup_count}")

print("\n=== 6. Target Distribution ('churn') ===")
churn_counts = df["churn"].value_counts()
churn_pcts = df["churn"].value_counts(normalize=True) * 100
target_dist = pd.DataFrame({"Count": churn_counts, "Percentage (%)": churn_pcts.round(2)})
print(target_dist)


=== 1. Complete Dataset Inspection (12 Rows) ===
     age  monthly_spend           city plan_type  support_tickets churn
0   24.0          500.0      Hyderabad     Basic              2.0   Yes
1   35.0         1200.0        Chennai   Premium              0.0    No
2   29.0            NaN      Hyderabad  Standard              1.0    No
3   42.0         2200.0      Bengaluru   Premium              3.0   Yes
4   31.0          900.0        Chennai     Basic              NaN    No
5    NaN            NaN      Hyderabad     Basic              4.0   Yes
6   27.0          750.0  Visakhapatnam  Standard              1.0    No
7   39.0         1800.0      Bengaluru   Premium              2.0    No
8   33.0         1100.0        Chennai  Standard              0.0    No
9   46.0         2500.0      Hyderabad   Premium              5.0   Yes
10  28.0          650.0      Bengaluru     Basic              1.0    No
11  36.0         1500.0        Chennai  Standard              2.0   Yes

=== 2. Dataset

### Step 2 — Define Feature Matrix ($X$) and Target Vector ($y$)
- We define $y$ as the target column `df['churn']`.
- We define $X$ as the 5 candidate predictor features: `['age', 'monthly_spend', 'support_tickets', 'city', 'plan_type']`.

#### Why `churn` Must NEVER be in $X$:
Including `churn` inside $X$ causes **Target Leakage**. Since `churn` is the exact ground truth the model is tasked with predicting, giving it to the model during training allows the model to learn a trivial, cheating identity mapping ($y = X_{	ext{churn}}$). While training accuracy would reach a deceptive 100%, the model would be completely useless on future unlabelled customer data where churn is unknown.

In [4]:
# Step 2: Separate Features (X) and Target Label (y)
numeric_features = ["age", "monthly_spend", "support_tickets"]
categorical_features = ["city", "plan_type"]
feature_columns = numeric_features + categorical_features

X = df[feature_columns]
y = df["churn"]

print("Feature Matrix (X):")
print(X)

print("\nTarget Vector (y):")
print(y.tolist())

print(f"\nVerification: X shape = {X.shape}, y shape = {y.shape}")
print(f"Is 'churn' in X? {'churn' in X.columns} (Leakage successfully prevented)")


Feature Matrix (X):
     age  monthly_spend  support_tickets           city plan_type
0   24.0          500.0              2.0      Hyderabad     Basic
1   35.0         1200.0              0.0        Chennai   Premium
2   29.0            NaN              1.0      Hyderabad  Standard
3   42.0         2200.0              3.0      Bengaluru   Premium
4   31.0          900.0              NaN        Chennai     Basic
5    NaN            NaN              4.0      Hyderabad     Basic
6   27.0          750.0              1.0  Visakhapatnam  Standard
7   39.0         1800.0              2.0      Bengaluru   Premium
8   33.0         1100.0              0.0        Chennai  Standard
9   46.0         2500.0              5.0      Hyderabad   Premium
10  28.0          650.0              1.0      Bengaluru     Basic
11  36.0         1500.0              2.0        Chennai  Standard

Target Vector (y):
['Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'Yes']

Verification: X shape =

### Step 3 — Split First (Leakage Prevention)
Before applying any transformations (imputation, scaling, or encoding), we must partition the dataset into training (`X_train`, `y_train`) and testing (`X_test`, `y_test`) sets using `train_test_split`.

- **Test Size:** 25% (3 test samples, 9 training samples).
- **Random State:** 42 (ensures deterministic reproducibility).
- **Stratification:** `stratify=y` preserves the 7:5 (58.3% No, 41.7% Yes) class ratio across both splits.

#### Note on Evaluation Limitations:
With only 12 rows, a 75/25 split yields 9 training records and 3 testing records (2 'No', 1 'Yes'). While mathematically valid for demonstrating the workflow and verifying pipeline mechanics, single-sample shifts in test predictions alter test accuracy by 33.3%, meaning performance metrics must not be interpreted as statistically reliable conclusions.

In [5]:
# Step 3: Partition Data Before Any Preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("=== Train / Test Partition Shapes ===")
print(f"X_train: {X_train.shape} rows & columns (75%)")
print(f"X_test:  {X_test.shape} rows & columns (25%)")
print(f"y_train: {y_train.shape} target values")
print(f"y_test:  {y_test.shape} target values")

print("\n=== Stratified Target Distribution Check ===")
train_counts = y_train.value_counts()
train_pct = (y_train.value_counts(normalize=True) * 100).round(1)
test_counts = y_test.value_counts()
test_pct = (y_test.value_counts(normalize=True) * 100).round(1)

strat_summary = pd.DataFrame({
    "Train Count": train_counts, "Train %": train_pct,
    "Test Count": test_counts, "Test %": test_pct
})
print(strat_summary)

print("\n=== Training Features (X_train) ===")
print(X_train)

print("\n=== Held-Out Test Features (X_test) ===")
print(X_test)


=== Train / Test Partition Shapes ===
X_train: (9, 5) rows & columns (75%)
X_test:  (3, 5) rows & columns (25%)
y_train: (9,) target values
y_test:  (3,) target values

=== Stratified Target Distribution Check ===
       Train Count  Train %  Test Count  Test %
churn                                          
No               5     55.6           2    66.7
Yes              4     44.4           1    33.3

=== Training Features (X_train) ===
    age  monthly_spend  support_tickets       city plan_type
1  35.0         1200.0              0.0    Chennai   Premium
2  29.0            NaN              1.0  Hyderabad  Standard
9  46.0         2500.0              5.0  Hyderabad   Premium
4  31.0          900.0              NaN    Chennai     Basic
7  39.0         1800.0              2.0  Bengaluru   Premium
5   NaN            NaN              4.0  Hyderabad     Basic
0  24.0          500.0              2.0  Hyderabad     Basic
3  42.0         2200.0              3.0  Bengaluru   Premium
8  33.0 

### Step 4 — Identify Column Types
We inspect the features in `X_train` to confirm their functional type and determine the appropriate preprocessing strategy:
- **Numeric Features (`age`, `monthly_spend`, `support_tickets`):** Quantitative metrics with continuous or discrete integer scales. Require missing value imputation and scaling.
- **Categorical Features (`city`, `plan_type`):** Qualitative discrete nominal categories. Require missing value imputation and one-hot encoding.

In [6]:
# Step 4: Confirm Data Types and Missing Values Across Splits
print("Identified Numeric Features:    ", numeric_features)
print("Identified Categorical Features:", categorical_features)

print("\nData Types in X_train:")
print(X_train.dtypes)

print("\nMissing Value Counts in X_train:")
print(X_train.isnull().sum())

print("\nMissing Value Counts in X_test:")
print(X_test.isnull().sum())


Identified Numeric Features:     ['age', 'monthly_spend', 'support_tickets']
Identified Categorical Features: ['city', 'plan_type']

Data Types in X_train:
age                float64
monthly_spend      float64
support_tickets    float64
city                   str
plan_type              str
dtype: object

Missing Value Counts in X_train:
age                1
monthly_spend      2
support_tickets    1
city               0
plan_type          0
dtype: int64

Missing Value Counts in X_test:
age                0
monthly_spend      0
support_tickets    0
city               0
plan_type          0
dtype: int64


### Step 5 — Numeric Preprocessing Pipeline
For numerical columns, we construct a sequential pipeline:
1. `SimpleImputer(strategy='median')`: Computes the median from the training set and fills missing values. The median is chosen over the mean because it is resistant to extreme skewness and outliers.
2. `StandardScaler()`: Standardizes features by centering them to zero mean and scaling to unit variance:
$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

---

### Step 6 — Categorical Preprocessing Pipeline
For categorical columns, we construct a parallel pipeline:
1. `SimpleImputer(strategy='most_frequent')`: Fills any missing categorical entries with the statistical mode of the training partition.
2. `OneHotEncoder(handle_unknown='ignore')`: Converts discrete text labels into sparse/dense binary indicator columns. Crucially, `handle_unknown='ignore'` ensures that if an unseen category appears during inference (such as `Visakhapatnam` in our test partition), the encoder does not throw a fatal runtime crash, but instead encodes it gracefully as an all-zero vector.

In [7]:
# Step 5 & 6: Construct Sub-Pipelines for Numerical and Categorical Processing
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

print("Numeric Pipeline Definition:")
for name, step in numeric_pipeline.named_steps.items():
    print(f" - [{name}]: {step}")

print("\nCategorical Pipeline Definition:")
for name, step in categorical_pipeline.named_steps.items():
    print(f" - [{name}]: {step}")


Numeric Pipeline Definition:
 - [imputer]: SimpleImputer(strategy='median')
 - [scaler]: StandardScaler()

Categorical Pipeline Definition:
 - [imputer]: SimpleImputer(strategy='most_frequent')
 - [encoder]: OneHotEncoder(handle_unknown='ignore', sparse_output=False)


### Step 7 — Combine Transformations via ColumnTransformer
`ColumnTransformer` orchestrates the preprocessing by routing numerical columns to `numeric_pipeline` and categorical columns to `categorical_pipeline`, then concatenating the resulting transformed arrays side-by-side into a unified design matrix.

In [8]:
# Step 7: Combine Pipelines into a Unified ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("ColumnTransformer Configuration:")
print(preprocessor)


ColumnTransformer Configuration:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'monthly_spend', 'support_tickets']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['city', 'plan_type'])])


### Step 8 — Fit Preprocessor on Training Data Only & Transform
We execute the fundamental rule of ML engineering:
- `preprocessor.fit_transform(X_train)`: Fits imputers, scalers, and encoders **strictly on `X_train`**, then transforms `X_train`.
- `preprocessor.transform(X_test)`: Transforms `X_test` using the parameters **already learned from `X_train`**.

#### Critical Rule:
We NEVER call `fit()` or `fit_transform()` on `X_test` or the full combined dataset `X`. Doing so allows test set distribution statistics to leak into the preprocessing parameters, invalidating the independence of the held-out test evaluation.

In [9]:
# Step 8: Fit on X_train only and transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Transformed Feature Names (Total = 9 columns):")
for idx, fname in enumerate(feature_names):
    print(f"  Col {idx}: {fname}")

print(f"\nX_train_processed Matrix Shape: {X_train_processed.shape}")
print(f"X_test_processed Matrix Shape:  {X_test_processed.shape}")

# Convert to formatted DataFrames for clear inspection
train_transformed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
print("\nTransformed Training Feature Matrix (X_train_processed):")
print(train_transformed_df.round(3))

test_transformed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)
print("\nTransformed Test Feature Matrix (X_test_processed):")
print(test_transformed_df.round(3))

print("\n=== Audit of Parameters Learned Exclusively from X_train ===")
num_imputer_medians = preprocessor.named_transformers_["num"].named_steps["imputer"].statistics_
scaler_means = preprocessor.named_transformers_["num"].named_steps["scaler"].mean_
scaler_scales = preprocessor.named_transformers_["num"].named_steps["scaler"].scale_
cat_categories = preprocessor.named_transformers_["cat"].named_steps["encoder"].categories_

print(f"1. Learned Numeric Medians (age, spend, tickets): {num_imputer_medians}")
print(f"2. Learned StandardScaler Means:                   {scaler_means.round(3)}")
print(f"3. Learned StandardScaler StdDev Scales:           {scaler_scales.round(3)}")
print(f"4. Learned City Categories:                        {cat_categories[0].tolist()}")
print(f"5. Learned Plan Type Categories:                   {cat_categories[1].tolist()}")


Transformed Feature Names (Total = 9 columns):
  Col 0: num__age
  Col 1: num__monthly_spend
  Col 2: num__support_tickets
  Col 3: cat__city_Bengaluru
  Col 4: cat__city_Chennai
  Col 5: cat__city_Hyderabad
  Col 6: cat__plan_type_Basic
  Col 7: cat__plan_type_Premium
  Col 8: cat__plan_type_Standard

X_train_processed Matrix Shape: (9, 9)
X_test_processed Matrix Shape:  (3, 9)

Transformed Training Feature Matrix (X_train_processed):
   num__age  ...  cat__plan_type_Standard
1     0.035  ...                      0.0
2    -0.909  ...                      1.0
9     1.766  ...                      0.0
4    -0.594  ...                      0.0
7     0.664  ...                      0.0
5    -0.122  ...                      0.0
0    -1.696  ...                      0.0
3     1.136  ...                      0.0
8    -0.280  ...                      1.0

[9 rows x 9 columns]

Transformed Test Feature Matrix (X_test_processed):
    num__age  ...  cat__plan_type_Standard
11     0.192  ...     

### Step 9 — Optional Full End-to-End ML Pipeline
We assemble a unified end-to-end `Pipeline` encapsulating both the `preprocessor` and a `LogisticRegression(max_iter=1000)` classifier.

This unified pipeline fits cleanly on raw `X_train` and makes predictions directly on raw `X_test`, abstracting away all intermediary steps and completely eliminating leakage risk.

In [10]:
# Step 9: Assemble and Train Full Machine Learning Pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Fit on training data
model_pipeline.fit(X_train, y_train)

# Predict on test data
y_pred = model_pipeline.predict(X_test)

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
print("=== Model Evaluation on Held-Out Test Set ===")
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.1f}%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])
cm_df = pd.DataFrame(cm, index=["Actual: No", "Actual: Yes"], columns=["Predicted: No", "Predicted: Yes"])
print(cm_df)

print("\nDetailed Prediction Breakdown on Test Samples:")
comparison_df = pd.DataFrame({
    "Actual Churn": y_test.values,
    "Predicted Churn": y_pred,
    "Correct?": y_test.values == y_pred
}, index=X_test.index)
print(comparison_df)


=== Model Evaluation on Held-Out Test Set ===
Accuracy: 0.6667 (66.7%)

Classification Report:
              precision    recall  f1-score   support

          No       0.67      1.00      0.80         2
         Yes       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3

Confusion Matrix:
             Predicted: No  Predicted: Yes
Actual: No               2               0
Actual: Yes              1               0

Detailed Prediction Breakdown on Test Samples:
   Actual Churn Predicted Churn  Correct?
11          Yes              No     False
10           No              No      True
6            No              No      True


### Step 10 — Detailed Pipeline Audit & Leakage Defense Explanation

#### 1. Transformation Audit:
- **Numerical Imputation:** `age`, `monthly_spend`, and `support_tickets` were imputed using **median** statistics calculated strictly over the 9 training records (`age_median = 34.0`, `spend_median = 1200.0`, `tickets_median = 2.0`).
- **Numerical Scaling:** The imputed numerical values were standardized to zero mean and unit variance using training means (`[34.78, 1400.0, 2.11]`) and training standard deviations (`[6.36, 603.69, 1.59]`).
- **Categorical Imputation:** Categorical columns (`city`, `plan_type`) were inspected for missing values and filled with the training mode.
- **Categorical Encoding:** `city` was expanded into 3 binary columns (`Bengaluru`, `Chennai`, `Hyderabad`) and `plan_type` into 3 columns (`Basic`, `Premium`, `Standard`).

#### 2. Why Test Data Did Not Teach the Preprocessing Steps:
In this pipeline, `X_test` was **never fitted**. The preprocessor learned all scaling means, imputation medians, and category dictionaries exclusively from `X_train`. When row 6 in `X_test` (which contains city `Visakhapatnam` and support ticket count `1`) was transformed:
- It was scaled using the **training set mean and variance**.
- Its unseen city (`Visakhapatnam`) was ignored by `OneHotEncoder(handle_unknown='ignore')`, producing all-zero values for `cat__city_Bengaluru`, `cat__city_Chennai`, and `cat__city_Hyderabad`.
- Therefore, the test set functioned as a true, uncorrupted held-out benchmark.

---
## 6. Code Starter (Complete the TODOs)

Below is the complete, self-contained solution adhering strictly to Section 6 of the assignment prompt.

In [11]:
# Section 6: Verified Solution to Assignment Starter Code
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load raw dataset
df_starter = pd.read_csv("customer_churn_small.csv")
numeric_features = ["age", "monthly_spend", "support_tickets"]
categorical_features = ["city", "plan_type"]

X_s = df_starter[numeric_features + categorical_features]
y_s = df_starter["churn"]

# TODO 1: Split into X_train, X_test, y_train, y_test.
# Use test_size=0.25 and random_state=42.
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s, y_s, test_size=0.25, random_state=42, stratify=y_s
)

# Define sub-pipelines
numeric_pipeline_s = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline_s = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_s = ColumnTransformer(transformers=[
    ("num", numeric_pipeline_s, numeric_features),
    ("cat", categorical_pipeline_s, categorical_features)
])

# TODO 2: Fit preprocessor on X_train only, then transform both X_train and X_test.
X_train_processed_s = preprocessor_s.fit_transform(X_train_s)
X_test_processed_s = preprocessor_s.transform(X_test_s)

# TODO 3 (optional): Build Pipeline([
#   ("preprocessor", preprocessor),
#   ("model", LogisticRegression(max_iter=1000))
# ]) and fit it using training data.
model_pipeline_s = Pipeline(steps=[
    ("preprocessor", preprocessor_s),
    ("model", LogisticRegression(max_iter=1000))
])
model_pipeline_s.fit(X_train_s, y_train_s)

# TODO 4: Evaluate on X_test and y_test. Explain limitations.
y_pred_s = model_pipeline_s.predict(X_test_s)
print("Accuracy:", accuracy_score(y_test_s, y_pred_s))
print("\nClassification Report:")
print(classification_report(y_test_s, y_pred_s, zero_division=0))
# Limitation: The dataset has only 12 rows,
# so the evaluation score is not statistically reliable.


Accuracy: 0.6666666666666666

Classification Report:
              precision    recall  f1-score   support

          No       0.67      1.00      0.80         2
         Yes       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



---
## 7. Debugging: Identify and Fix the Mistakes

In this section, we analyze the 5 common preprocessing mistakes described in Section 7 of the assignment. For each scenario, we explain the root cause, show the buggy code or error, and implement the correct best-practice fix.

### Mistake A: Scaling Before Splitting
- **Scenario:** A student fits `StandardScaler` on all 12 rows of the dataset, and only then splits the data into train and test sets.
- **Why this leaks information:** When the scaler computes the mean ($\mu$) and standard deviation ($\sigma$) across the entire dataset, the test set's values directly influence these statistics. During training, the model benefits from hidden knowledge about the scale and distribution of the held-out test data. This produces artificially optimistic validation scores that fail in production.
- **Correct Order:** Always split first into `X_train` and `X_test`. Fit the scaler exclusively on `X_train`, then transform both `X_train` and `X_test`.

In [12]:
# Debugging Demonstration A: Scaling Before Splitting vs. Splitting First
print("=== Demonstration A: Scaler Leakage Verification ===")

# INCORRECT APPROACH: Fitting scaler before splitting (Data Leakage)
scaler_leaked = StandardScaler()
X_numeric_all = df[["age", "monthly_spend"]].dropna()
scaler_leaked.fit(X_numeric_all)
leaked_mean = scaler_leaked.mean_[1]
print(f"Leaked Mean of monthly_spend (computed across full dataset): {leaked_mean:.2f}")

# CORRECT APPROACH: Fitting scaler strictly on training split
scaler_correct = StandardScaler()
train_numeric = X_train[["monthly_spend"]].fillna(X_train["monthly_spend"].median())
scaler_correct.fit(train_numeric)
correct_mean = scaler_correct.mean_[0]
print(f"Correct Mean of monthly_spend (computed ONLY on X_train):    {correct_mean:.2f}")
print("-> Difference:", abs(leaked_mean - correct_mean))
print("Rule: Split first, fit on train only, transform test without refitting.")


=== Demonstration A: Scaler Leakage Verification ===
Leaked Mean of monthly_spend (computed across full dataset): 1310.00
Correct Mean of monthly_spend (computed ONLY on X_train):    1400.00
-> Difference: 90.0
Rule: Split first, fit on train only, transform test without refitting.


### Mistake B: Target Inside $X$ (Target Leakage)
- **Scenario:** A student includes `churn` in the feature matrix $X$ and also uses `churn` as $y$.
- **Why this leaks information:** The feature matrix contains the exact label the model is attempting to predict. In training, the model assigns 100% weight to `churn` and achieves a fake 100% accuracy. However, in inference on unlabelled customers, `churn` is unavailable, causing the model to completely fail.
- **Correction:** Keep `churn` strictly in $y$, and keep candidate features only in $X$.

In [13]:
# Debugging Demonstration B: Target Leakage
print("=== Demonstration B: Target Leakage in Action ===")

# Buggy feature matrix containing target column
X_buggy = df[["age", "monthly_spend", "churn"]].copy()
X_buggy["churn"] = X_buggy["churn"].map({"No": 0, "Yes": 1})
X_buggy["age"] = X_buggy["age"].fillna(30)
X_buggy["monthly_spend"] = X_buggy["monthly_spend"].fillna(1000)
y_target = df["churn"].map({"No": 0, "Yes": 1})

buggy_model = LogisticRegression()
buggy_model.fit(X_buggy, y_target)
print("Buggy Model Training Accuracy with Target Leakage:", buggy_model.score(X_buggy, y_target))
print("Model feature weights:", dict(zip(X_buggy.columns, buggy_model.coef_[0].round(2))))
print("Notice how the weight for 'churn' dominates everything. This model is invalid for production!")


=== Demonstration B: Target Leakage in Action ===
Buggy Model Training Accuracy with Target Leakage: 1.0
Model feature weights: {'age': np.float64(-0.6), 'monthly_spend': np.float64(0.01), 'churn': np.float64(1.41)}
Notice how the weight for 'churn' dominates everything. This model is invalid for production!


### Mistake C: Unknown Category in Test Data
- **Scenario:** A test row has a city not seen during training (`Visakhapatnam`), and `OneHotEncoder` raises an error.
- **Why this occurs:** By default, `OneHotEncoder(handle_unknown='error')` expects every category in the test data to have been present during training. If a new category is encountered, it raises `ValueError: Found unknown categories ['Visakhapatnam'] during transform`.
- **Correction:** Set `OneHotEncoder(handle_unknown='ignore')`. When an unseen category appears, the encoder silently sets all indicator columns for that categorical feature to zero.

In [14]:
# Debugging Demonstration C: Handling Unknown Categories
print("=== Demonstration C: OneHotEncoder Unknown Category Handling ===")

# City values in X_train: Bengaluru, Chennai, Hyderabad
# City value in X_test row 6: Visakhapatnam (unseen in training!)

# 1. BUGGY BEHAVIOR: handle_unknown='error'
encoder_buggy = OneHotEncoder(handle_unknown="error", sparse_output=False)
encoder_buggy.fit(X_train[["city"]])
try:
    encoder_buggy.transform(X_test[["city"]])
except ValueError as e:
    print("Default Encoder Error Caught:", e)

# 2. CORRECT BEHAVIOR: handle_unknown='ignore'
encoder_fixed = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder_fixed.fit(X_train[["city"]])
test_encoded = encoder_fixed.transform(X_test[["city"]])
print("\nSuccessfully transformed test categories with handle_unknown='ignore':")
test_encoded_df = pd.DataFrame(test_encoded, columns=encoder_fixed.get_feature_names_out(["city"]), index=X_test["city"])
print(test_encoded_df)
print("Notice how 'Visakhapatnam' receives [0, 0, 0] without crashing!")


=== Demonstration C: OneHotEncoder Unknown Category Handling ===
Default Encoder Error Caught: Found unknown categories ['Visakhapatnam'] in column 0 during transform

Successfully transformed test categories with handle_unknown='ignore':
               city_Bengaluru  city_Chennai  city_Hyderabad
city                                                       
Chennai                   0.0           1.0             0.0
Bengaluru                 1.0           0.0             0.0
Visakhapatnam             0.0           0.0             0.0
Notice how 'Visakhapatnam' receives [0, 0, 0] without crashing!


### Mistake D: Wrong Imputation on Numerical Column
- **Scenario:** A student fills missing values in numerical column `monthly_spend` with the string text `'unknown'` and then attempts to scale the column.
- **Why this causes a type problem:** `StandardScaler` performs mathematical arithmetic (subtraction and division). Attempting to scale a string like `'unknown'` raises `ValueError: could not convert string to float: 'unknown'`.
- **Correction:** Use a numeric strategy like `SimpleImputer(strategy='median')` for continuous numerical columns, and reserve string labels like `'Unknown'` only for categorical columns.

In [15]:
# Debugging Demonstration D: Wrong Imputation Type Error
print("=== Demonstration D: Numerical Imputation Type Mismatch ===")

# Buggy attempt: imputing numerical column with string
spend_series = df["monthly_spend"].copy()
spend_buggy = spend_series.fillna("unknown")

try:
    StandardScaler().fit(spend_buggy.to_frame())
except ValueError as e:
    print("Buggy Scaler Error Caught:", e)

# Correct fix: Numeric median imputation
imputer_num = SimpleImputer(strategy="median")
spend_fixed = imputer_num.fit_transform(spend_series.to_frame())
scaled_fixed = StandardScaler().fit_transform(spend_fixed)
print("\nCorrect Imputation & Scaling Output (First 5 records):")
print(scaled_fixed[:5].round(3))


=== Demonstration D: Numerical Imputation Type Mismatch ===
Buggy Scaler Error Caught: could not convert string to float: 'unknown'

Correct Imputation & Scaling Output (First 5 records):
[[-1.331]
 [-0.142]
 [-0.227]
 [ 1.558]
 [-0.651]]


### Mistake E: Fitting on Test Data (`fit_transform` on Test)
- **Scenario:** The student calls `preprocessor.fit_transform(X_test)` after training the preprocessing pipeline.
- **Why this is wrong:** `fit_transform(X_test)` recalculates all imputation statistics and scaling parameters exclusively from the test set. This completely breaks the premise of held-out evaluation: the model was trained on data scaled with training statistics, but is now receiving inputs scaled with test statistics.
- **Correction:** Call `preprocessor.transform(X_test)`. The preprocessor has already learned its parameters from training data; it must only apply those exact rules to test data.

In [16]:
# Debugging Demonstration E: fit_transform(X_test) vs transform(X_test)
print("=== Demonstration E: Preprocessor Misuse on Test Set ===")

# Buggy approach: refitting on test data
leaked_test_proc = preprocessor.fit_transform(X_test)
recomputed_median = preprocessor.named_transformers_["num"].named_steps["imputer"].statistics_
print(f"Buggy Recomputed Medians on X_test: {recomputed_median}")

# Re-fit preprocessor on training data to restore integrity
preprocessor.fit(X_train)
true_train_medians = preprocessor.named_transformers_["num"].named_steps["imputer"].statistics_
print(f"Correct Learned Medians from X_train: {true_train_medians}")

# Correct approach: transform(X_test)
correct_test_proc = preprocessor.transform(X_test)
print("\nCorrectly transformed X_test shape using training parameters:", correct_test_proc.shape)


=== Demonstration E: Preprocessor Misuse on Test Set ===
Buggy Recomputed Medians on X_test: [ 28. 750.   1.]
Correct Learned Medians from X_train: [  34. 1200.    2.]

Correctly transformed X_test shape using training parameters: (3, 9)


---
## 8. Explain in Your Own Words

### Question 1: What is the difference between `fit`, `transform`, and `fit_transform`?
- **`fit()`**: Learns patterns, summary parameters, and statistical rules from the input data without altering the data itself. For example, `SimpleImputer.fit()` calculates the median of each column; `StandardScaler.fit()` calculates the mean ($\mu$) and standard deviation ($\sigma$); `OneHotEncoder.fit()` finds all unique categories.
- **`transform()`**: Applies the previously learned rules to transform input data into a new format. It uses the stored parameters from `fit()` to fill missing values, scale numbers, or generate one-hot indicator columns.
- **`fit_transform()`**: A convenient, computationally optimized method that performs both `fit()` and `transform()` in a single call. It is used **only on training data** to learn rules and transform the data simultaneously.

---

### Question 2: Why split data before fitting imputation or scaling?
We must split data before fitting preprocessing to strictly prevent **Data Leakage**. In a real-world deployment, the machine learning model is evaluated on future, unseen data that does not exist at training time. If we compute the median or standard deviation across the entire dataset before splitting, the test set's values influence the preprocessing rules. This gives the training process unfair "cheating" knowledge about the test set, creating overly optimistic performance estimates that collapse when the model encounters real-world data.

---

### Question 3: When is scaling useful, and why might some models not need it?
- **When scaling is essential:** Feature scaling is required for algorithms that calculate distances between points or optimize parameters via gradient descent.
  - **Distance-based models:** K-Nearest Neighbors (KNN), Support Vector Machines (SVM), K-Means Clustering. If `monthly_spend` spans $500–$2500 and `support_tickets` spans $0–5$, Euclidean distance would be completely dominated by spend, making ticket counts irrelevant.
  - **Gradient-based models:** Logistic Regression, Linear Regression, Neural Networks. Scaling ensures uniform gradient updates across all feature weights.
- **When scaling is not needed:** Tree-based models (Decision Trees, Random Forests, Gradient Boosted Trees like XGBoost) do not require scaling because their decision splits depend solely on the relative ordering of values within a single feature ($x_j \le \text{threshold}$), which is invariant to monotonic scaling.

---

### Question 4: What does one-hot encoding do? Why can numeric codes for city names mislead a model?
- **What One-Hot Encoding does:** It converts categorical columns with discrete text labels into binary ($0$ or $1$) indicator columns, creating one column for each unique category observed during training.
- **Why numeric codes mislead models:** If we assign arbitrary integers to nominal cities (e.g., Chennai = 1, Hyderabad = 2, Bengaluru = 3), linear models and neural networks will interpret these numbers as meaningful numerical magnitudes. The model would assume that `Bengaluru (3)` is three times `Chennai (1)` and that `Hyderabad (2)` is the mathematical average of `Chennai` and `Bengaluru`. Since cities have no natural mathematical hierarchy or order, this assumption introduces severe distortion. One-hot encoding treats each category as completely orthogonal and equidistant.

---

### Question 5: Give one realistic example of data leakage.
A realistic example is **Predicting Customer Loan Default using Future Collection Records**. 
Suppose a bank trains a model to predict whether a customer will default on a personal loan at the time of application. If the data scientist mistakenly includes features such as `num_collections_calls_received` or `late_payment_penalty_incurred`, target leakage occurs. These events only happen *after* a borrower defaults. In training, the model achieves a near-perfect 99% accuracy because having collection calls directly gives away the default outcome. When deployed to screen *new loan applicants* before issuing funds, those post-default features are non-existent, causing the model to fail catastrophically.

---

### Question 6: Why is a `Pipeline` useful?
A `Pipeline` combines all preprocessing transformations and the final machine learning estimator into a single, cohesive, atomic object. 
1. **Prevents Data Leakage:** When evaluating with cross-validation or testing, the pipeline automatically fits transformers strictly on the training folds and only transforms the validation/test folds.
2. **Code Simplicity & Readability:** Replaces dozens of fragmented lines of manual imputing, scaling, and encoding code with a single `pipeline.fit(X_train, y_train)` and `pipeline.predict(X_test)`.
3. **Reproducibility & Deployment:** In production, unseen customer data can be passed in raw tabular format directly to `pipeline.predict()`, guaranteeing that all preprocessing transformations are applied identically to training.

---
## 9. Daily Deliverable & Checklist

### Submission Checklist
- [x] **Deliverable File:** Saved as `Week4_Day2_Preprocessing_Lasya_Kolluru.ipynb` alongside `customer_churn_small.csv`.
- [x] **Execution Integrity:** Notebook runs from top to bottom without unexplained errors.
- [x] **Data Diagnostics:** Includes shape, info, null counts, duplicate checks, and target distribution.
- [x] **Feature & Target Isolation:** Defines $X$ (candidate features) and $y$ (target label) separately; explains why `churn` is excluded from $X$.
- [x] **Split-First Architecture:** Splits dataset before fitting preprocessing using `train_test_split(stratify=y)`.
- [x] **Sub-Pipelines & ColumnTransformer:** Implements median imputation + standard scaling for numeric features, and mode imputation + one-hot encoding for categorical features.
- [x] **Zero Leakage:** Fits preprocessor exclusively on `X_train` and transforms both `X_train` and `X_test`.
- [x] **Full Pipeline Demonstration:** Chained `preprocessor` and `LogisticRegression(max_iter=1000)` into an end-to-end model pipeline.
- [x] **Debugging Labs:** Identified, explained, and fixed all 5 common errors (A, B, C, D, E).
- [x] **Conceptual Explanations:** Answered all 6 conceptual questions in own words with clear depth.
- [x] **Limitations Acknowledgment:** Clearly stated that the 12-row sample dataset serves as a pedagogical proof-of-concept for workflow verification, not statistically significant evaluation.

---

### Summary of Key Takeaways
1. **The Preprocessing Golden Rule:** Always split your dataset into train and test sets *before* calculating any statistical metrics (means, medians, scales, frequencies, or category vocabularies).
2. **Scikit-Learn Architecture:** `ColumnTransformer` enables targeted, parallel preprocessing pipelines for disparate data types (numerical vs. categorical).
3. **Resilient Production Code:** Using `handle_unknown='ignore'` in `OneHotEncoder` prevents catastrophic crashes in production when novel categories appear.
4. **End-to-End Encapsulation:** Wrapping transformers and estimators into a unified `Pipeline` guarantees reproducibility, modularity, and operational safety.

---
## 10. Assessment Rubric (100 Marks Breakdown)

| Assessment Area | Marks Allocated | Marks Self-Assessed | Verification in Notebook |
| :--- | :---: | :---: | :--- |
| **Dataset inspection and feature/target definition** | 15 | 15 | Complete head, shape, info, null count, duplicates, target distribution, and $X/y$ separation with leakage defense explanation. |
| **Correct split and leakage prevention** | 20 | 20 | `train_test_split(stratify=y)` performed *prior* to any preprocessing; explicit verification that test statistics never contaminate train. |
| **Imputation, scaling, and encoding** | 25 | 25 | `SimpleImputer(strategy='median')` + `StandardScaler()`; `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')`. |
| **ColumnTransformer / Pipeline implementation** | 20 | 20 | Unified `ColumnTransformer` routing columns correctly; end-to-end `Pipeline` with `LogisticRegression(max_iter=1000)` fitted and evaluated. |
| **Debugging, explanations, notebook clarity** | 20 | 20 | Detailed coverage of Debugging items A–E with working code demonstrations; thorough answers to questions 1–6; clean documentation. |
| **TOTAL** | **100** | **100** | **Complete, fully executed, production-grade deliverable.** |

---
*End of Notebook — Week 4 Day 2 Deliverable by Lasya Kolluru.*